## Run SUMMA model ##

Run the SUMMA base model, with the model runs managed by Snakemake


In [1]:

from pathlib import Path

import sys
sys.path.append('../scripts')
from ss_utils import resolve_paths

In [ ]:
config_file = "/Users/dcasson/GitHub/snow_dist/snakemake/config/chena_above_two_rivers.yaml"

### Write snakemake file ###

This block of code writes the snakemake file

In [9]:
%%writefile ../rules/run_base_summa_simulation.smk
"""

Snakemake file to run the base SUMMA simulations.

The model simulation is chunked by GRU to allow for parallelization on a cluster.

The chunks of GRUs are defined by the user

"""

from pathlib import Path
import sys

# Import local packages
sys.path.append(str(Path('../').resolve()))
from scripts import ss_utils

# Resolve all file paths and directories in the config file
config['attributes_nc'] = '/Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/settings/attributes.nc'
config['gru_chunk_size'] = 3
config['run_suffix'] = ''
config['file_manager'] = '/Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/settings/fileManager.txt'
config['summa_exe'] = '/Users/drc858/GitHub/summa/bin/summa.exe'
config['summa_output_dir'] = '/Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/output/'
config['summa_settings_dir'] = '/Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/settings/fileManager.txt'

# Generate GRU start and count
num_grus = ss_utils.calc_num_grus(config['attributes_nc'])
gru_chunk_strings = ss_utils.generate_gru_start_and_count(num_grus, chunk_size=config['gru_chunk_size'])

rule run_summa_base_simulations:
    input:
        expand(Path(config['summa_output_dir'],"tuolumne_{gru_chunk}_timestep.nc"),gru_chunk=gru_chunk_strings)

rule run_summa_in_gru_chunks:
    input:
        file_manager = Path(config['summa_settings_dir'],config['file_manager'])
    output:
        summa_chunked_output = Path(config['summa_output_dir'],"tuolumne_{gru_chunk}_timestep.nc")
    params:
        summa_exe = config['summa_exe'],
        gru_start = lambda wildcards: ss_utils.extract_gru_int({wildcards.gru_chunk}),
        gru_count = config['gru_chunk_size']
    shell:
        """
        {params.summa_exe} -m {input.file_manager}
        """

        

Overwriting ../rules/run_base_summa_simulation.smk


In [10]:
! snakemake --unlock -s ../rules/run_base_summa_simulation.smk --configfile /Users/drc858/GitHub/summa_snakemake/snakemake/config/summa_snakemake_config_tuolumne.yaml 
! snakemake -s ../rules/run_base_summa_simulation.smk --cores 8 --configfile /Users/drc858/GitHub/summa_snakemake/snakemake/config/summa_snakemake_config_tuolumne.yaml


['/Users/drc858/GitHub/snow_dist/snakemake/rules', '/Users/drc858/.pyenv/versions/3.9.16/envs/gmet_to_summa_snakemake/bin', '/Users/drc858/PycharmProjects/snow_dist/lib/python3.9/site-packages/osgeo', '/Library/Frameworks/GDAL.framework/Programs', '/Users/drc858/GitHub/snow_dist/snakemake/notebooks', '/Users/drc858/.pyenv/versions/3.9.16/lib/python39.zip', '/Users/drc858/.pyenv/versions/3.9.16/lib/python3.9', '/Users/drc858/.pyenv/versions/3.9.16/lib/python3.9/lib-dynload', '/Users/drc858/.pyenv/versions/3.9.16/envs/gmet_to_summa_snakemake/lib/python3.9/site-packages', '/opt/gurobi201/linux32/lib/python2.5', '/Users/drc858/GitHub/snow_dist/snakemake']
Unlocking working directory.
['/Users/drc858/GitHub/snow_dist/snakemake/rules', '/Users/drc858/.pyenv/versions/3.9.16/envs/gmet_to_summa_snakemake/bin', '/Users/drc858/PycharmProjects/snow_dist/lib/python3.9/site-packages/osgeo', '/Library/Frameworks/GDAL.framework/Programs', '/Users/drc858/GitHub/snow_dist/snakemake/notebooks', '/Users/d

In [1]:
%%writefile ../rules/run_fsca_simulations.smk
"""

Snakemake file to calculate fSCA based on SUMMA model simulations.

"""

from pathlib import Path
import sys
# Import local packages
sys.path.append(str(Path('../').resolve()))
sys.path.append(str(Path('../../').resolve()))
sys.path.append(str(Path('../../snow_dist').resolve()))
from scripts import ss_utils
from snow_dist import summa_simulation

# Resolve all file paths and directories in the config file
config['summa_output_dir'] = '/Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/output'
config['fsca_output_dir'] =  '/Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/output/fsca/'
config_file_path = '/Users/drc858/GitHub/snow_dist/settings/config_summa_model_tuolumne.yaml'

#Read first raw forcing file for easymore remapping
summa_result_files = list(Path(config['summa_output_dir']).glob('*.nc'))
print(summa_result_files)

fsca_simulation = summa_simulation.SummaSimulation(config_file_path)

#Initialize fsca model run
rule run_fsca_simulations:
    input:
        expand(Path(config['fsca_output_dir'], "{summa_files}"), output_files=summa_result_files)

rule run_fsca_in_gru_chunks:
    input:
        input_summa_result = Path(config['summa_output_dir'],"{summa_files}")
    output:
        fsca_output = expand(Path(config['fsca_output_dir'], "{summa_files}"), output_files=summa_result_files)
    run:
        fsca_simulation.settings[["summa"]["summa_fsca_filename"]] = output.fsca_output
        fsca_simulation.run_summa_fsca_model(input.input_summa_result)


Overwriting ../rules/run_fsca_simulations.smk


In [2]:
! snakemake --unlock -s ../rules/run_fsca_simulations.smk --configfile /Users/drc858/GitHub/summa_snakemake/snakemake/config/summa_snakemake_config_tuolumne.yaml 
! snakemake -s ../rules/run_fsca_simulations.smk --cores 8 --configfile /Users/drc858/GitHub/summa_snakemake/snakemake/config/summa_snakemake_config_tuolumne.yaml


/Users/drc858/GitHub/snow_dist/snakemake/rules/run_fsca_simulations.smk:52: SyntaxWarning: list indices must be integers or slices, not str; perhaps you missed a comma?
[PosixPath('/Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/output/tuolumne_H134_timestep.nc'), PosixPath('/Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/output/tuolumne_001_H51_timestep.nc'), PosixPath('/Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/output/tuolumne_timestep_backup.nc'), PosixPath('/Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/output/tuolumne_006_H51_timestep.nc'), PosixPath('/Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/output/tuolumne_009_H51_timestep.nc'), PosixPath('/Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/output/tuolumne_010_H51_timestep.nc'), PosixPath('/Users/drc858/Data/summa_snakemake/hydrofabric_tuolumne/summa/output/tuolumne_007_H51_timestep.nc'), PosixPath('/Users/drc858/Data/summa_snakemake/hydr

In [ ]:
model_dist.run()

### Run snakemake file

In [ ]:
! snakemake --unlock -s ../build_summa_model.smk --configfile /Users/drc858/GitHub/summa_snakemake/snakemake/config/summa_snakemake_config.yaml 
! snakemake -s ../build_summa_model.smk --cores 1 --configfile /Users/drc858/GitHub/summa_snakemake/snakemake/config/summa_snakemake_config.yaml --debug

In [ ]:
from IPython import display
# Snake Command line to build the image
! snakemake -s ../build_summa_model.smk --configfile /Users/drc858/GitHub/summa_snakemake/snakemake/config/summa_snakemake_config.yaml --filegraph | dot -Tpng > ../reports/make_summa_model.png
# Python command to visualise the built image in our notebook
display.Image('../reports/make_summa_model.png')